In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

In [2]:
df = pd.read_csv('prices_all.csv')

In [3]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['Ticker', 'date']).reset_index(drop=True)

In [4]:
print(df.head())

        date Ticker      Open      High       Low     Close        Volume
0 2008-01-02   AAPL  5.976315  6.006006  5.774775  5.843454  1.079179e+09
1 2008-01-03   AAPL  5.860551  5.919933  5.778975  5.846155  8.420664e+08
2 2008-01-04   AAPL  5.741786  5.788272  5.365099  5.399888  1.455832e+09
3 2008-01-07   AAPL  5.435876  5.506355  5.105375  5.327609  2.072193e+09
4 2008-01-08   AAPL  5.402587  5.472167  5.122471  5.135967  1.523816e+09


### Поделю на train и test

In [5]:
train_list, test_list = [], []

for ticker, group in df.groupby('Ticker'):
    if len(group) < 300:
        continue
    
    split_date = group['date'].quantile(0.8)
    train_list.append(group[group['date'] < split_date])
    test_list.append(group[group['date'] >= split_date])

train = pd.concat(train_list, ignore_index=True)
test = pd.concat(test_list, ignore_index=True)

### Добавлю target (доходность)

In [6]:
train['target'] = np.log(train['Close'].shift(-1) / train['Close'])
test['target'] = np.log(test['Close'].shift(-1) / test['Close'])

### Добавлю базовые дневные показатели (доходность)

In [ ]:
train['high_low_ratio'] = train['High'] / train['Low']  # Отношение максимума к минимуму
train['high_low_spread'] = train['High'] - train['Low']  # Размах за день
train['close_open_ratio'] = train['Close'] / train['Open']  # Отношение закрытия к открытию

test['high_low_ratio'] = test['High'] / test['Low']
test['high_low_spread'] = test['High'] - test['Low']
test['close_open_ratio'] = test['Close'] / test['Open']

### Добавлю:
- определение дней с высокой волатильностью, 
- определение бычьего/медвежьего дня, 
- базовый признак волатильности, 
- сглаженную волатильность
- нормализованную волатильность.

In [ ]:
def calculate_true_range(group):
    high_low = group['High'] - group['Low']
    high_close = abs(group['High'] - group['Close'].shift(1))
    low_close = abs(group['Low'] - group['Close'].shift(1))
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr

def calculate_atr(group, period=14):
    tr = calculate_true_range(group)
    atr = tr.ewm(span=period, adjust=False).mean()
    return atr

train['true_range'] = train.groupby('Ticker', group_keys=False).apply(calculate_true_range)
train['atr_14'] = train.groupby('Ticker', group_keys=False).apply(lambda x: calculate_atr(x, 14))
train['atr_ratio_14'] = train['atr_14'] / train['Close']

test['true_range'] = test.groupby('Ticker', group_keys=False).apply(calculate_true_range)
test['atr_14'] = test.groupby('Ticker', group_keys=False).apply(lambda x: calculate_atr(x, 14))
test['atr_ratio_14'] = test['atr_14'] / df['Close']

In [10]:
train = train.dropna().reset_index(drop=True)
test = test.dropna().reset_index(drop=True)

### Добавлю признак: скользящие средние от логарифма цены на 20, 50, 250 дней

In [11]:
log_close_shift = np.log(train['Close'].shift(1))

for window in [20, 50, 250]:
    train[f'ma_log_{window}'] = log_close_shift.rolling(window).mean()

log_close_shift = np.log(test['Close'].shift(1))

for window in [20, 50, 250]:
    test[f'ma_log_{window}'] = log_close_shift.rolling(window).mean()

### Дальше добавлю доходности за предыдущие дни

In [13]:
for lag in [1, 2, 3, 5]:
    train[f'return_lag_{lag}'] = np.log(train['Close'].shift(lag) / train['Close'].shift(lag + 1))

for lag in [1, 2, 3, 5]:
    test[f'return_lag_{lag}'] = np.log(test['Close'].shift(lag) / test['Close'].shift(lag + 1))

### Дальше добавлю историческую волатильность

In [14]:
return_1d = np.log(train['Close'].shift(1) / train['Close'].shift(2))
train['volatility'] = return_1d.rolling(20).std().shift(1)

return_1d = np.log(test['Close'].shift(1) / test['Close'].shift(2))
test['volatility'] = return_1d.rolling(20).std().shift(1)

### Дальше добавлю дневной диапазон

In [15]:
train['daily_range'] = (train['High'].shift(1) - train['Low'].shift(1)) / train['Close'].shift(1)

test['daily_range'] = (test['High'].shift(1) - test['Low'].shift(1)) / test['Close'].shift(1)

### Дальше добавлю ценовые разницы

In [16]:
train['high_open_diff'] = (train['High'].shift(1) - train['Open'].shift(1)) / train['Open'].shift(1)
train['open_low_diff'] = (train['Open'].shift(1) - train['Low'].shift(1)) / train['Low'].shift(1)

test['high_open_diff'] = (test['High'].shift(1) - test['Open'].shift(1)) / test['Open'].shift(1)
test['open_low_diff'] = (test['Open'].shift(1) - test['Low'].shift(1)) / test['Low'].shift(1)

### Дальше добавлю (вдруг поможет) временные признаки раздельно

In [17]:
train['day_of_week'] = train['date'].dt.dayofweek
train['month'] = train['date'].dt.month

test['day_of_week'] = test['date'].dt.dayofweek
test['month'] = test['date'].dt.month

### Удалю все что с nan

In [29]:
train = train.dropna().reset_index(drop=True)
test = test.dropna().reset_index(drop=True)


### Подготовка к обучению (выделю признаки), маштабирование и сплит

In [ ]:
feat =  ['high_low_ratio', 'high_low_spread', 'close_open_ratio', 'true_range', 'atr_14', 'atr_ratio_14', 'ma_log_20', 'ma_log_50', 'ma_log_250','return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5',
        'volatility', 'daily_range', 'high_open_diff', 'open_low_diff', 'day_of_week', 'month']

X_train = train[feat]
y_train = train['target']
X_test = test[feat]
y_test = test['target']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Обучение моделей

In [23]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred_lr)):.4f}")

R²: -0.0091
RMSE: 0.0527
MAE: 0.0150


In [27]:
model = Ridge(alpha=0.01)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred)):.4f}")

R²: -0.0091
RMSE: 0.0527
MAE: 0.0150


In [24]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10]
lasso_results = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    y_pred_lasso = lasso.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred_lasso)
    lasso_results.append({'alpha': alpha, 'R²': r2})


best_alpha = max(lasso_results, key=lambda x: x['R²'])['alpha']

lasso_best = Lasso(alpha=best_alpha, max_iter=10000)
lasso_best.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_best.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred_lasso):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred_lasso)):.4f}")

R²: 0.0001
RMSE: 0.0524
MAE: 0.0145


In [28]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
en_results = []

for alpha in alphas:
    for l1_ratio in l1_ratios:
        en = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
        en.fit(X_train_scaled, y_train)
        y_pred_en = en.predict(X_test_scaled)
        r2 = r2_score(y_test, y_pred_en)
        en_results.append({'alpha': alpha, 'l1_ratio': l1_ratio, 'R²': r2})

best_en = max(en_results, key=lambda x: x['R²'])

en_best = ElasticNet(alpha=best_en['alpha'], l1_ratio=best_en['l1_ratio'], max_iter=10000)
en_best.fit(X_train_scaled, y_train)
y_pred_en = en_best.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred_en):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_en)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred_en)):.4f}")

R²: 0.0006
RMSE: 0.0524
MAE: 0.0146


### Выводы:
- Модели предсказивают плохо (ниже чем просто среднее)
- ElasticNet показыват лучшую метрику среди других линейных моделей (но все равно плохую)

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

# Загрузка кластеров
cluster_df = pd.read_csv('cluster_fullstart_assignments.csv')

# Фильтрация тикеров с >= 500 записей (как в N-HiTS)
ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()
df_filtered = df[df['Ticker'].isin(valid_tickers)]

# Добавление кластеров (как в N-HiTS)
df_filtered = df_filtered.merge(cluster_df[['Company', 'Cluster']], 
                                 left_on='Ticker', 
                                 right_on='Company', 
                                 how='left')
df_filtered['cluster_id'] = df_filtered['Cluster'].fillna(-1).astype(int)
cluster_dummies = pd.get_dummies(df_filtered['cluster_id'], prefix='cluster')
df_filtered = pd.concat([df_filtered, cluster_dummies], axis=1)
cluster_features = [col for col in df_filtered.columns if col.startswith('cluster_')]

# Разделение на train/test (как в N-HiTS)
TRAIN_END = '2024-12-31'
TEST_START = '2025-01-01'

train_df = df_filtered[df_filtered['date'] <= TRAIN_END].copy()
test_df = df_filtered[(df_filtered['date'] >= TEST_START)].copy()

print(f"Train: {len(train_df)} записей")
print(f"Test: {len(test_df)} записей")
print(f"Тикеров в train: {train_df['Ticker'].nunique()}")
print(f"Тикеров в test: {test_df['Ticker'].nunique()}")

Train: 777845 записей
Test: 48348 записей
Тикеров в train: 199
Тикеров в test: 199


In [2]:
def create_features_for_linear(df):
    """Создает признаки для линейных моделей (предсказание цены)"""
    df = df.copy()
    
    # Сортируем по тикеру и дате
    df = df.sort_values(['Ticker', 'date'])
    
    # Базовые признаки (как в N-HiTS, но адаптированные для цены)
    df['high_low_ratio'] = df['High'] / df['Low']
    df['high_low_spread'] = df['High'] - df['Low']
    df['close_open_ratio'] = df['Close'] / df['Open']
    
    # Лаги цены (последние N дней)
    for lag in [1, 2, 3, 5, 10, 20]:
        df[f'close_lag_{lag}'] = df.groupby('Ticker')['Close'].shift(lag)
    
    # Скользящие средние
    for window in [5, 10, 20, 50, 100]:
        df[f'ma_{window}'] = df.groupby('Ticker')['Close'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    # Скользящие стандартные отклонения (волатильность)
    for window in [5, 10, 20]:
        df[f'std_{window}'] = df.groupby('Ticker')['Close'].transform(
            lambda x: x.rolling(window, min_periods=1).std()
        )
    
    # Отношение цены к скользящей средней
    for window in [5, 10, 20, 50]:
        df[f'price_to_ma_{window}'] = df['Close'] / df[f'ma_{window}']
    
    # Дневной диапазон (нормализованный)
    df['daily_range_pct'] = (df['High'] - df['Low']) / df['Close']
    
    # Изменение цены за день
    df['price_change'] = df.groupby('Ticker')['Close'].pct_change()
    
    # Лаги доходности
    for lag in [1, 2, 3, 5]:
        df[f'return_lag_{lag}'] = df.groupby('Ticker')['Close'].pct_change(lag)
    
    # Кластерные признаки (как в N-HiTS)
    cluster_cols = [col for col in df.columns if col.startswith('cluster_')]
    
    # Удаляем строки с NaN
    feature_cols = ['high_low_ratio', 'high_low_spread', 'close_open_ratio', 
                    'daily_range_pct', 'price_change'] + \
                   [f'close_lag_{lag}' for lag in [1, 2, 3, 5, 10, 20]] + \
                   [f'ma_{window}' for window in [5, 10, 20, 50, 100]] + \
                   [f'std_{window}' for window in [5, 10, 20]] + \
                   [f'price_to_ma_{window}' for window in [5, 10, 20, 50]] + \
                   [f'return_lag_{lag}' for lag in [1, 2, 3, 5]] + \
                   cluster_cols
    
    # Удаляем строки с NaN в признаках
    df_clean = df[feature_cols + ['Close', 'Ticker', 'date']].dropna()
    
    return df_clean, feature_cols

# Создаем признаки
train_features, feature_cols = create_features_for_linear(train_df)
test_features, _ = create_features_for_linear(test_df)

print(f"Train features shape: {train_features.shape}")
print(f"Test features shape: {test_features.shape}")
print(f"Количество признаков: {len(feature_cols)}")

Train features shape: (773865, 38)
Test features shape: (44368, 38)
Количество признаков: 35


In [3]:
X_train = train_features[feature_cols]
y_train = train_features['Close']

X_test = test_features[feature_cols]
y_test = test_features['Close']

# Масштабирование (как в N-HiTS использовался robust scaler)
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_test shape: {X_test_scaled.shape}")
print(f"Mean Close: ${y_train.mean():.2f}")
print(f"Std Close: ${y_train.std():.2f}")

X_train shape: (773865, 35)
X_test shape: (44368, 35)
Mean Close: $95.79
Std Close: $186.01


In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Обучает и оценивает модель"""
    # Обучение
    model.fit(X_train, y_train)
    
    # Предсказания
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Метрики
    metrics = {
        'Model': model_name,
        'Train R²': r2_score(y_train, y_pred_train),
        'Test R²': r2_score(y_test, y_pred_test),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Train MAE': mean_absolute_error(y_train, y_pred_train),
        'Test MAE': mean_absolute_error(y_test, y_pred_test),
        'Test MAE $': mean_absolute_error(y_test, y_pred_test),
    }
    
    # WAPE (как в N-HiTS)
    def wape(y_true, y_pred):
        denominator = np.sum(np.abs(y_true))
        if denominator == 0:
            return np.nan
        return 100.0 * np.sum(np.abs(y_true - y_pred)) / denominator
    
    metrics['Test WAPE'] = wape(y_test, y_pred_test)
    
    return metrics, y_pred_test

# Список моделей для обучения
models = {
    'Linear Regression': LinearRegression(),
    #'Ridge (α=0.01)': Ridge(alpha=0.01),
    #'Ridge (α=0.1)': Ridge(alpha=0.1),
    #'Ridge (α=1.0)': Ridge(alpha=1.0),
    #'Lasso (α=0.0001)': Lasso(alpha=0.0001, max_iter=10000),
    #'Lasso (α=0.001)': Lasso(alpha=0.001, max_iter=10000),
    #'Lasso (α=0.01)': Lasso(alpha=0.01, max_iter=10000),
    #'ElasticNet (α=0.001, l1=0.5)': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000),
    #'ElasticNet (α=0.01, l1=0.5)': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
}

# Обучение всех моделей
results = []
predictions = {}

for name, model in models.items():
    print(f"Обучение {name}...")
    metrics, y_pred = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test, name)
    results.append(metrics)
    predictions[name] = y_pred

# Сводная таблица результатов
results_df = pd.DataFrame(results)
print("\n" + "="*100)
print("РЕЗУЛЬТАТЫ ЛИНЕЙНЫХ МОДЕЛЕЙ (ПРЕДСКАЗАНИЕ ЦЕНЫ)")
print("="*100)
print(results_df[['Model', 'Test R²', 'Test RMSE', 'Test MAE $', 'Test WAPE']].to_string(index=False))

Обучение Linear Regression...

РЕЗУЛЬТАТЫ ЛИНЕЙНЫХ МОДЕЛЕЙ (ПРЕДСКАЗАНИЕ ЦЕНЫ)
            Model  Test R²  Test RMSE  Test MAE $  Test WAPE
Linear Regression 0.999873   5.365051    1.795939    0.65514


In [ ]:
# Результаты N-HiTS (из вашего эксперимента)
nhits_results = {
    'Model': 'N-HiTS (base_model)',
    'Test R²': np.nan,  # N-HiTS не считал R²
    'Test RMSE': np.nan,  # N-HiTS не считал RMSE
    'Test MAE $': 12.55,
    'Test WAPE': 5.16,
}

# Добавляем N-HiTS в таблицу сравнения
comparison_df = results_df[['Model', 'Test R²', 'Test RMSE', 'Test MAE $', 'Test WAPE']].copy()
comparison_df = pd.concat([comparison_df, pd.DataFrame([nhits_results])], ignore_index=True)

print("\n" + "="*100)
print("СРАВНЕНИЕ ЛИНЕЙНЫХ МОДЕЛЕЙ vs N-HiTS")
print("="*100)
print(comparison_df.to_string(index=False))

# Находим лучшую линейную модель
best_linear = results_df.loc[results_df['Test MAE $'].idxmin()]
print("\n" + "="*100)
print(f"ЛУЧШАЯ ЛИНЕЙНАЯ МОДЕЛЬ: {best_linear['Model']}")
print(f"  Test MAE: ${best_linear['Test MAE $']:.2f}")
print(f"  Test WAPE: {best_linear['Test WAPE']:.2f}%")
print("="*100)

# Сравнение с N-HiTS
print("\n" + "="*100)
print("СРАВНЕНИЕ С N-HiTS:")
print(f"  N-HiTS MAE: ${nhits_results['Test MAE $']:.2f}")
print(f"  Лучшая Linear MAE: ${best_linear['Test MAE $']:.2f}")
print(f"  Разница: ${abs(nhits_results['Test MAE $'] - best_linear['Test MAE $']):.2f}")

if best_linear['Test MAE $'] < nhits_results['Test MAE $']:
    improvement = (nhits_results['Test MAE $'] - best_linear['Test MAE $']) / nhits_results['Test MAE $'] * 100
    print(f"  ✅ Линейная модель ЛУЧШЕ на {improvement:.1f}%")
else:
    improvement = (best_linear['Test MAE $'] - nhits_results['Test MAE $']) / best_linear['Test MAE $'] * 100
    print(f"  ✅ N-HiTS ЛУЧШЕ на {improvement:.1f}%")
print("="*100)

In [ ]:
import matplotlib.pyplot as plt

# Сравнение MAE
fig, ax = plt.subplots(figsize=(12, 6))
models_for_plot = results_df['Model'].tolist() + ['N-HiTS']
mae_values = results_df['Test MAE $'].tolist() + [nhits_results['Test MAE $']]
wape_values = results_df['Test WAPE'].tolist() + [nhits_results['Test WAPE']]

x = np.arange(len(models_for_plot))
width = 0.35

bars1 = ax.bar(x - width/2, mae_values, width, label='MAE ($)', color='steelblue')
bars2 = ax.bar(x + width/2, wape_values, width, label='WAPE (%)', color='coral')

ax.set_xlabel('Модель')
ax.set_ylabel('Значение')
ax.set_title('Сравнение линейных моделей и N-HiTS')
ax.set_xticks(x)
ax.set_xticklabels(models_for_plot, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# Добавляем значения на график
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'${height:.1f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = best_linear['Model']
best_model = models[best_model_name]

# Получаем коэффициенты
if hasattr(best_model, 'coef_'):
    coefficients = best_model.coef_
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Coefficient': coefficients,
        'Abs_Coefficient': np.abs(coefficients)
    }).sort_values('Abs_Coefficient', ascending=False)
    
    print("\n" + "="*100)
    print(f"ТОП-20 ВАЖНЕЙШИХ ПРИЗНАКОВ ({best_model_name})")
    print("="*100)
    print(feature_importance.head(20).to_string(index=False))

In [6]:
def wape(y_true, y_pred):
        denominator = np.sum(np.abs(y_true))
        if denominator == 0:
            return np.nan
        return 100.0 * np.sum(np.abs(y_true - y_pred)) / denominator
    
train_features['target_30d'] = train_features.groupby('Ticker')['Close'].shift(-30)
test_features['target_30d'] = test_features.groupby('Ticker')['Close'].shift(-30)

# Удаляем NaN
train_30d = train_features.dropna(subset=['target_30d'])
test_30d = test_features.dropna(subset=['target_30d'])

# Обучаем Linear Regression
X_train_30d = train_30d[feature_cols]
y_train_30d = train_30d['target_30d']
X_test_30d = test_30d[feature_cols]
y_test_30d = test_30d['target_30d']

# Масштабируем
scaler_30d = RobustScaler()
X_train_30d_scaled = scaler_30d.fit_transform(X_train_30d)
X_test_30d_scaled = scaler_30d.transform(X_test_30d)

# Обучаем
lr_30d = LinearRegression()
lr_30d.fit(X_train_30d_scaled, y_train_30d)
y_pred_30d = lr_30d.predict(X_test_30d_scaled)

# Метрики
mae_30d = mean_absolute_error(y_test_30d, y_pred_30d)
wape_30d = wape(y_test_30d, y_pred_30d)

print(f"Linear Regression (30 дней):")
print(f"  MAE: ${mae_30d:.2f}")
print(f"  WAPE: {wape_30d:.2f}%")
print(f"  R²: {r2_score(y_test_30d, y_pred_30d):.4f}")

Linear Regression (30 дней):
  MAE: $21.90
  WAPE: 7.93%
  R²: 0.9899


In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных (как в предыдущем коде)
df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

# Фильтрация и подготовка (как в N-HiTS)
cluster_df = pd.read_csv('cluster_fullstart_assignments.csv')
ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()
df_filtered = df[df['Ticker'].isin(valid_tickers)]

df_filtered = df_filtered.merge(cluster_df[['Company', 'Cluster']], 
                                 left_on='Ticker', 
                                 right_on='Company', 
                                 how='left')
df_filtered['cluster_id'] = df_filtered['Cluster'].fillna(-1).astype(int)
cluster_dummies = pd.get_dummies(df_filtered['cluster_id'], prefix='cluster')
df_filtered = pd.concat([df_filtered, cluster_dummies], axis=1)
cluster_features = [col for col in df_filtered.columns if col.startswith('cluster_')]

# Разделение на train/test
TRAIN_END = '2024-12-31'
TEST_START = '2025-01-01'

train_df = df_filtered[df_filtered['date'] <= TRAIN_END].copy()
test_df = df_filtered[df_filtered['date'] >= TEST_START].copy()

print(f"Train: {len(train_df)} записей")
print(f"Test: {len(test_df)} записей")

Train: 777845 записей
Test: 48348 записей


In [8]:
def create_features_for_linear(df):
    """Создает признаки для линейных моделей"""
    df = df.copy()
    df = df.sort_values(['Ticker', 'date'])
    
    # Базовые признаки
    df['high_low_ratio'] = df['High'] / df['Low']
    df['high_low_spread'] = df['High'] - df['Low']
    df['close_open_ratio'] = df['Close'] / df['Open']
    
    # Лаги цены
    for lag in [1, 2, 3, 5, 10, 20, 50, 100]:
        df[f'close_lag_{lag}'] = df.groupby('Ticker')['Close'].shift(lag)
    
    # Скользящие средние
    for window in [5, 10, 20, 50, 100, 200]:
        df[f'ma_{window}'] = df.groupby('Ticker')['Close'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    # Волатильность
    for window in [5, 10, 20, 50]:
        df[f'std_{window}'] = df.groupby('Ticker')['Close'].transform(
            lambda x: x.rolling(window, min_periods=1).std()
        )
    
    # Отношение цены к скользящей средней
    for window in [5, 10, 20, 50, 100]:
        df[f'price_to_ma_{window}'] = df['Close'] / df[f'ma_{window}']
    
    # Дневной диапазон
    df['daily_range_pct'] = (df['High'] - df['Low']) / df['Close']
    df['price_change'] = df.groupby('Ticker')['Close'].pct_change()
    
    # Лаги доходности
    for lag in [1, 2, 3, 5, 10]:
        df[f'return_lag_{lag}'] = df.groupby('Ticker')['Close'].pct_change(lag)
    
    # Кластерные признаки
    cluster_cols = [col for col in df.columns if col.startswith('cluster_')]
    
    # Список всех признаков
    feature_cols = ['high_low_ratio', 'high_low_spread', 'close_open_ratio', 
                    'daily_range_pct', 'price_change'] + \
                   [f'close_lag_{lag}' for lag in [1, 2, 3, 5, 10, 20, 50, 100]] + \
                   [f'ma_{window}' for window in [5, 10, 20, 50, 100, 200]] + \
                   [f'std_{window}' for window in [5, 10, 20, 50]] + \
                   [f'price_to_ma_{window}' for window in [5, 10, 20, 50, 100]] + \
                   [f'return_lag_{lag}' for lag in [1, 2, 3, 5, 10]] + \
                   cluster_cols
    
    return df, feature_cols

# Создаем признаки для train и test
train_features, feature_cols = create_features_for_linear(train_df)
test_features, _ = create_features_for_linear(test_df)

print(f"Количество признаков: {len(feature_cols)}")

Количество признаков: 41


In [9]:
def evaluate_horizon(train_df, test_df, feature_cols, horizon_days, horizon_name):
    """
    Обучает Linear Regression для заданного горизонта
    """
    print(f"\n{'='*80}")
    print(f"ГОРИЗОНТ: {horizon_name} ({horizon_days} дней)")
    print(f"{'='*80}")
    
    # Создаем целевую переменную (цена через N дней)
    train_df[f'target_{horizon_days}d'] = train_df.groupby('Ticker')['Close'].shift(-horizon_days)
    test_df[f'target_{horizon_days}d'] = test_df.groupby('Ticker')['Close'].shift(-horizon_days)
    
    # Удаляем NaN
    train_clean = train_df.dropna(subset=[f'target_{horizon_days}d'])
    test_clean = test_df.dropna(subset=[f'target_{horizon_days}d'])
    
    # Подготовка данных
    X_train = train_clean[feature_cols]
    y_train = train_clean[f'target_{horizon_days}d']
    X_test = test_clean[feature_cols]
    y_test = test_clean[f'target_{horizon_days}d']
    
    # Масштабирование
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Обучение
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # Метрики
    def wape(y_true, y_pred):
        denominator = np.sum(np.abs(y_true))
        if denominator == 0:
            return np.nan
        return 100.0 * np.sum(np.abs(y_true - y_pred)) / denominator
    
    metrics = {
        'Horizon': horizon_name,
        'Days': horizon_days,
        'R²': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred),
        'WAPE': wape(y_test, y_pred),
        'Train Size': len(X_train),
        'Test Size': len(X_test)
    }
    
    print(f"R²:  {metrics['R²']:.4f}")
    print(f"RMSE: ${metrics['RMSE']:.2f}")
    print(f"MAE:  ${metrics['MAE']:.2f}")
    print(f"WAPE: {metrics['WAPE']:.2f}%")
    print(f"Train samples: {metrics['Train Size']}")
    print(f"Test samples: {metrics['Test Size']}")
    
    return metrics, model, scaler, y_test, y_pred

# Список горизонтов для тестирования
horizons = [
    (1, '1 день'),
    (7, '1 неделя'),
    (30, '1 месяц'),
    (90, '3 месяца'),
    (180, '6 месяцев'),
    (364, '1 год (недели)'),  # 52 недели
    (365, '1 год (месяцы)'),  # 12 месяцев
]

# Обучаем модели для всех горизонтов
results = []
models_dict = {}

for days, name in horizons:
    metrics, model, scaler, y_test, y_pred = evaluate_horizon(
        train_features.copy(), 
        test_features.copy(), 
        feature_cols, 
        days, 
        name
    )
    results.append(metrics)
    models_dict[name] = {
        'model': model,
        'scaler': scaler,
        'y_test': y_test,
        'y_pred': y_pred
    }

# Сводная таблица
results_df = pd.DataFrame(results)
print("\n" + "="*100)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ LINEAR REGRESSION")
print("="*100)
print(results_df[['Horizon', 'Days', 'R²', 'RMSE', 'MAE', 'WAPE']].to_string(index=False))


ГОРИЗОНТ: 1 день (1 дней)


ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

def evaluate_horizon(train_df, test_df, feature_cols, horizon_days, horizon_name):
    """
    Обучает Linear Regression для заданного горизонта
    """
    print(f"\n{'='*80}")
    print(f"ГОРИЗОНТ: {horizon_name} ({horizon_days} дней)")
    print(f"{'='*80}")
    
    # Копируем данные, чтобы не изменять оригиналы
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # Создаем целевую переменную (цена через N дней)
    train_df[f'target_{horizon_days}d'] = train_df.groupby('Ticker')['Close'].shift(-horizon_days)
    test_df[f'target_{horizon_days}d'] = test_df.groupby('Ticker')['Close'].shift(-horizon_days)
    
    # Удаляем строки с NaN в целевой переменной
    train_clean = train_df.dropna(subset=[f'target_{horizon_days}d'])
    test_clean = test_df.dropna(subset=[f'target_{horizon_days}d'])
    
    # Проверяем, что данные не пустые
    if len(train_clean) == 0 or len(test_clean) == 0:
        print(f"❌ Нет данных для горизонта {horizon_days} дней!")
        return None, None, None, None, None
    
    # Проверяем наличие NaN в признаках
    print(f"Проверка NaN в train: {train_clean[feature_cols].isna().sum().sum()} пропусков")
    print(f"Проверка NaN в test: {test_clean[feature_cols].isna().sum().sum()} пропусков")
    
    # Удаляем строки с NaN в признаках
    train_clean = train_clean.dropna(subset=feature_cols)
    test_clean = test_clean.dropna(subset=feature_cols)
    
    print(f"После удаления NaN: train={len(train_clean)}, test={len(test_clean)}")
    
    # Если после удаления NaN данных太少, пропускаем
    if len(train_clean) < 100 or len(test_clean) < 10:
        print(f"❌ Слишком мало данных: train={len(train_clean)}, test={len(test_clean)}")
        return None, None, None, None, None
    
    # Подготовка данных
    X_train = train_clean[feature_cols]
    y_train = train_clean[f'target_{horizon_days}d']
    X_test = test_clean[feature_cols]
    y_test = test_clean[f'target_{horizon_days}d']
    
    # Проверка на бесконечные значения
    if not np.isfinite(X_train).all().all():
        print("❌ Обнаружены бесконечные значения в X_train!")
        # Заменяем inf на NaN и удаляем
        X_train = X_train.replace([np.inf, -np.inf], np.nan)
        X_test = X_test.replace([np.inf, -np.inf], np.nan)
        train_clean = train_clean.dropna(subset=feature_cols)
        test_clean = test_clean.dropna(subset=feature_cols)
        X_train = train_clean[feature_cols]
        y_train = train_clean[f'target_{horizon_days}d']
        X_test = test_clean[feature_cols]
        y_test = test_clean[f'target_{horizon_days}d']
    
    # Масштабирование
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Обучение
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # Метрики
    def wape(y_true, y_pred):
        denominator = np.sum(np.abs(y_true))
        if denominator == 0:
            return np.nan
        return 100.0 * np.sum(np.abs(y_true - y_pred)) / denominator
    
    metrics = {
        'Horizon': horizon_name,
        'Days': horizon_days,
        'R²': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred),
        'WAPE': wape(y_test, y_pred),
        'Train Size': len(X_train),
        'Test Size': len(X_test)
    }
    
    print(f"R²:  {metrics['R²']:.4f}")
    print(f"RMSE: ${metrics['RMSE']:.2f}")
    print(f"MAE:  ${metrics['MAE']:.2f}")
    print(f"WAPE: {metrics['WAPE']:.2f}%")
    print(f"Train samples: {metrics['Train Size']}")
    print(f"Test samples: {metrics['Test Size']}")
    
    return metrics, model, scaler, y_test, y_pred

In [11]:
horizons = [
    (1, '1 день'),
    (7, '1 неделя'),
    (30, '1 месяц'),
    (90, '3 месяца'),
    (180, '6 месяцев'),
    (364, '1 год (52 недели)'),
    (365, '1 год (12 месяцев)'),
]

# Обучаем модели для всех горизонтов
results = []
models_dict = {}

for days, name in horizons:
    result = evaluate_horizon(
        train_features.copy(), 
        test_features.copy(), 
        feature_cols, 
        days, 
        name
    )
    
    if result[0] is not None:  # Если модель обучилась
        metrics, model, scaler, y_test, y_pred = result
        results.append(metrics)
        models_dict[name] = {
            'model': model,
            'scaler': scaler,
            'y_test': y_test,
            'y_pred': y_pred
        }
    else:
        print(f"⚠️ Пропускаем горизонт {name}")

# Сводная таблица
if results:
    results_df = pd.DataFrame(results)
    print("\n" + "="*100)
    print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ LINEAR REGRESSION")
    print("="*100)
    print(results_df[['Horizon', 'Days', 'R²', 'RMSE', 'MAE', 'WAPE']].to_string(index=False))
else:
    print("❌ Нет результатов для отображения")


ГОРИЗОНТ: 1 день (1 дней)
Проверка NaN в train: 43183 пропусков
Проверка NaN в test: 43183 пропусков
После удаления NaN: train=757746, test=28249
R²:  0.9996
RMSE: $10.27
MAE:  $3.96
WAPE: 1.39%
Train samples: 757746
Test samples: 28249

ГОРИЗОНТ: 1 неделя (7 дней)
Проверка NaN в train: 43183 пропусков
Проверка NaN в test: 43183 пропусков
После удаления NaN: train=756552, test=27055
R²:  0.9975
RMSE: $24.62
MAE:  $10.11
WAPE: 3.55%
Train samples: 756552
Test samples: 27055

ГОРИЗОНТ: 1 месяц (30 дней)
Проверка NaN в train: 43183 пропусков
Проверка NaN в test: 43183 пропусков
После удаления NaN: train=751975, test=22478
R²:  0.9905
RMSE: $47.89
MAE:  $21.23
WAPE: 7.39%
Train samples: 751975
Test samples: 22478

ГОРИЗОНТ: 3 месяца (90 дней)
Проверка NaN в train: 43183 пропусков
Проверка NaN в test: 43183 пропусков
После удаления NaN: train=740035, test=10538
R²:  0.9668
RMSE: $86.75
MAE:  $38.82
WAPE: 13.38%
Train samples: 740035
Test samples: 10538

ГОРИЗОНТ: 6 месяцев (180 дней)
Прове

In [12]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

# Загрузка кластеров
cluster_df = pd.read_csv('cluster_fullstart_assignments.csv')

# Фильтрация тикеров с >= 500 записей
ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()
df_filtered = df[df['Ticker'].isin(valid_tickers)]

# Добавление кластеров
df_filtered = df_filtered.merge(cluster_df[['Company', 'Cluster']], 
                                 left_on='Ticker', 
                                 right_on='Company', 
                                 how='left')
df_filtered['cluster_id'] = df_filtered['Cluster'].fillna(-1).astype(int)
cluster_dummies = pd.get_dummies(df_filtered['cluster_id'], prefix='cluster')
df_filtered = pd.concat([df_filtered, cluster_dummies], axis=1)
cluster_features = [col for col in df_filtered.columns if col.startswith('cluster_')]

# НОВОЕ РАЗДЕЛЕНИЕ: 2024 год - тест!
TRAIN_END = '2023-12-31'  # Все данные до 2024 года
TEST_START = '2024-01-01'
TEST_END = '2024-12-31'

train_df = df_filtered[df_filtered['date'] <= TRAIN_END].copy()
test_df = df_filtered[(df_filtered['date'] >= TEST_START) & (df_filtered['date'] <= TEST_END)].copy()

print(f"Train: {len(train_df)} записей ({train_df['date'].min()} - {train_df['date'].max()})")
print(f"Test: {len(test_df)} записей ({test_df['date'].min()} - {test_df['date'].max()})")
print(f"Тикеров в train: {train_df['Ticker'].nunique()}")
print(f"Тикеров в test: {test_df['Ticker'].nunique()}")

Train: 727697 записей (2008-01-02 00:00:00 - 2023-12-29 00:00:00)
Test: 50148 записей (2024-01-02 00:00:00 - 2024-12-31 00:00:00)
Тикеров в train: 199
Тикеров в test: 199


In [21]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

# Загрузка кластеров
cluster_df = pd.read_csv('cluster_fullstart_assignments.csv')

# Фильтрация тикеров
ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()
df_filtered = df[df['Ticker'].isin(valid_tickers)]

# Добавление кластеров
df_filtered = df_filtered.merge(cluster_df[['Company', 'Cluster']], 
                                 left_on='Ticker', 
                                 right_on='Company', 
                                 how='left')
df_filtered['cluster_id'] = df_filtered['Cluster'].fillna(-1).astype(int)
cluster_dummies = pd.get_dummies(df_filtered['cluster_id'], prefix='cluster')
df_filtered = pd.concat([df_filtered, cluster_dummies], axis=1)
cluster_features = [col for col in df_filtered.columns if col.startswith('cluster_')]

# НОВОЕ РАЗДЕЛЕНИЕ:
# Train: до конца 2023 года
# Test: 2024 год
# Данные 2025 года используются для создания целевой переменной!
TRAIN_END = '2023-12-31'
TEST_START = '2024-01-01'
TEST_END = '2024-12-31'

train_df = df_filtered[df_filtered['date'] <= TRAIN_END].copy()
test_df = df_filtered[(df_filtered['date'] >= TEST_START) & (df_filtered['date'] <= TEST_END)].copy()

print(f"Train: {len(train_df)} записей ({train_df['date'].min()} - {train_df['date'].max()})")
print(f"Test: {len(test_df)} записей ({test_df['date'].min()} - {test_df['date'].max()})")

# Проверяем наличие данных после теста (для создания целевой переменной)
data_after_test = df_filtered[df_filtered['date'] > TEST_END]
print(f"Данные после теста: {len(data_after_test)} записей")
print(f"Период: {data_after_test['date'].min()} - {data_after_test['date'].max()}")

# Проверяем максимальный горизонт, который можно предсказать
max_horizon = (data_after_test['date'].max() - test_df['date'].max()).days
print(f"Максимальный горизонт прогноза: {max_horizon} дней")

Train: 727697 записей (2008-01-02 00:00:00 - 2023-12-29 00:00:00)
Test: 50148 записей (2024-01-02 00:00:00 - 2024-12-31 00:00:00)
Данные после теста: 48348 записей
Период: 2025-01-02 00:00:00 - 2025-12-19 00:00:00
Максимальный горизонт прогноза: 353 дней


In [22]:
def create_features_for_linear(df):
    """Создает признаки для линейных моделей"""
    df = df.copy()
    df = df.sort_values(['Ticker', 'date'])
    
    # Базовые признаки
    df['high_low_ratio'] = df['High'] / df['Low']
    df['high_low_spread'] = df['High'] - df['Low']
    df['close_open_ratio'] = df['Close'] / df['Open']
    
    # Лаги цены
    for lag in [1, 2, 3, 5, 10, 20, 50, 100]:
        df[f'close_lag_{lag}'] = df.groupby('Ticker')['Close'].shift(lag)
    
    # Скользящие средние
    for window in [5, 10, 20, 50, 100, 200]:
        df[f'ma_{window}'] = df.groupby('Ticker')['Close'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    # Волатильность
    for window in [5, 10, 20, 50]:
        df[f'std_{window}'] = df.groupby('Ticker')['Close'].transform(
            lambda x: x.rolling(window, min_periods=1).std()
        )
    
    # Отношение цены к скользящей средней
    for window in [5, 10, 20, 50, 100]:
        df[f'price_to_ma_{window}'] = df['Close'] / df[f'ma_{window}']
    
    # Дневной диапазон
    df['daily_range_pct'] = (df['High'] - df['Low']) / df['Close']
    df['price_change'] = df.groupby('Ticker')['Close'].pct_change()
    
    # Лаги доходности
    for lag in [1, 2, 3, 5, 10]:
        df[f'return_lag_{lag}'] = df.groupby('Ticker')['Close'].pct_change(lag)
    
    # Кластерные признаки
    cluster_cols = [col for col in df.columns if col.startswith('cluster_')]
    
    # Список всех признаков
    feature_cols = ['high_low_ratio', 'high_low_spread', 'close_open_ratio', 
                    'daily_range_pct', 'price_change'] + \
                   [f'close_lag_{lag}' for lag in [1, 2, 3, 5, 10, 20, 50, 100]] + \
                   [f'ma_{window}' for window in [5, 10, 20, 50, 100, 200]] + \
                   [f'std_{window}' for window in [5, 10, 20, 50]] + \
                   [f'price_to_ma_{window}' for window in [5, 10, 20, 50, 100]] + \
                   [f'return_lag_{lag}' for lag in [1, 2, 3, 5, 10]] + \
                   cluster_cols
    
    return df, feature_cols

# Создаем признаки
train_features, feature_cols = create_features_for_linear(train_df)
test_features, _ = create_features_for_linear(test_df)

print(f"Количество признаков: {len(feature_cols)}")
print(f"Train shape: {train_features.shape}")
print(f"Test shape: {test_features.shape}")

Количество признаков: 41
Train shape: (727697, 50)
Test shape: (50148, 50)


In [27]:
def evaluate_horizon_v2(train_df, test_df, feature_cols, horizon_days, horizon_name, full_df):
    """
    Обучает Linear Regression для заданного горизонта (альтернативная версия)
    """
    print(f"\n{'='*80}")
    print(f"ГОРИЗОНТ: {horizon_name} ({horizon_days} дней)")
    print(f"{'='*80}")
    
    # Копируем данные
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # Создаем целевую переменную для train
    train_df[f'target_{horizon_days}d'] = train_df.groupby('Ticker')['Close'].shift(-horizon_days)
    train_clean = train_df.dropna(subset=[f'target_{horizon_days}d'])
    
    if len(train_clean) == 0:
        print(f"❌ Нет данных для train на горизонте {horizon_days} дней!")
        return None, None, None, None, None
    
    # Для тестовых данных: создаем дату через horizon_days дней
    test_df['future_date'] = test_df['date'] + pd.Timedelta(days=horizon_days)
    
    # Ограничиваем future_prices только нужными датами
    future_dates = test_df['future_date'].unique()
    future_prices = full_df[full_df['date'].isin(future_dates)][['Ticker', 'date', 'Close']].copy()
    future_prices = future_prices.rename(columns={'date': 'future_date', 'Close': 'future_close'})
    
    # Merge (теперь намного меньше данных!)
    test_merged = test_df.merge(
        future_prices,
        on=['Ticker', 'future_date'],
        how='inner'
    )
    
    if len(test_merged) == 0:
        print(f"❌ Нет данных для test на горизонте {horizon_days} дней!")
        return None, None, None, None, None
    
    # Удаляем NaN в признаках
    train_clean = train_clean.dropna(subset=feature_cols)
    test_clean = test_merged.dropna(subset=feature_cols)
    
    print(f"После удаления NaN: train={len(train_clean)}, test={len(test_clean)}")
    
    if len(train_clean) < 100 or len(test_clean) < 10:
        print(f"❌ Слишком мало данных: train={len(train_clean)}, test={len(test_clean)}")
        return None, None, None, None, None
    
    # Подготовка данных
    X_train = train_clean[feature_cols]
    y_train = train_clean[f'target_{horizon_days}d']
    X_test = test_clean[feature_cols]
    y_test = test_clean['future_close']
    
    # Проверка на бесконечные значения
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    
    # Удаляем строки с NaN в признаках
    valid_train = ~X_train.isna().any(axis=1)
    valid_test = ~X_test.isna().any(axis=1)
    
    X_train = X_train[valid_train]
    y_train = y_train[valid_train]
    X_test = X_test[valid_test]
    y_test = y_test[valid_test]
    
    print(f"После удаления NaN в признаках: train={len(X_train)}, test={len(X_test)}")
    
    if len(X_train) < 100 or len(X_test) < 10:
        print(f"❌ Слишком мало данных после очистки!")
        return None, None, None, None, None
    
    # Масштабирование
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Обучение
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # Метрики
    def wape(y_true, y_pred):
        denominator = np.sum(np.abs(y_true))
        if denominator == 0:
            return np.nan
        return 100.0 * np.sum(np.abs(y_true - y_pred)) / denominator
    
    metrics = {
        'Horizon': horizon_name,
        'Days': horizon_days,
        'R²': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred),
        'WAPE': wape(y_test, y_pred),
        'Train Size': len(X_train),
        'Test Size': len(X_test)
    }
    
    print(f"R²:  {metrics['R²']:.4f}")
    print(f"RMSE: ${metrics['RMSE']:.2f}")
    print(f"MAE:  ${metrics['MAE']:.2f}")
    print(f"WAPE: {metrics['WAPE']:.2f}%")
    print(f"Train samples: {metrics['Train Size']}")
    print(f"Test samples: {metrics['Test Size']}")
    
    return metrics, model, scaler, y_test, y_pred

In [28]:
# Используем оптимизированную версию
for days, name in horizons:
    result = evaluate_horizon_v2(
        train_features.copy(), 
        test_features.copy(), 
        feature_cols, 
        days, 
        name,
        df_filtered
    )
    
    if result[0] is not None:
        metrics, model, scaler, y_test, y_pred = result
        results.append(metrics)
        models_dict[name] = {
            'model': model,
            'scaler': scaler,
            'y_test': y_test,
            'y_pred': y_pred
        }
    else:
        print(f"⚠️ Пропускаем горизонт {name}")


ГОРИЗОНТ: 1 день (1 дней)
После удаления NaN: train=707598, test=22885
После удаления NaN в признаках: train=707598, test=22885
R²:  0.9995
RMSE: $8.95
MAE:  $3.41
WAPE: 1.39%
Train samples: 707598
Test samples: 22885

ГОРИЗОНТ: 1 неделя (7 дней)
После удаления NaN: train=706404, test=29253
После удаления NaN в признаках: train=706404, test=29253
R²:  0.9980
RMSE: $17.99
MAE:  $7.29
WAPE: 2.97%
Train samples: 706404
Test samples: 29253

ГОРИЗОНТ: 1 месяц (30 дней)
После удаления NaN: train=701827, test=17114
После удаления NaN в признаках: train=701827, test=17114
R²:  0.9910
RMSE: $38.85
MAE:  $15.43
WAPE: 6.21%
Train samples: 701827
Test samples: 17114

ГОРИЗОНТ: 3 месяца (90 дней)
После удаления NaN: train=689887, test=22885
После удаления NaN в признаках: train=689887, test=22885
R²:  0.9753
RMSE: $67.11
MAE:  $27.13
WAPE: 10.63%
Train samples: 689887
Test samples: 22885

ГОРИЗОНТ: 6 месяцев (180 дней)
После удаления NaN: train=671977, test=16915
После удаления NaN в признаках: tr